# Кольцевые коммуникации

Реализуйте три операции на CPU с Gloo. 
- `rank` — номер процесса,
- `world_size` — число процессов.
- Входные тензоры — одномерные CPU-тензоры одинакового размера на всех процессах. Их менять нельзя

In [ ]:
%%writefile collectives.py
import torch
import torch.distributed as dist


def ring_reduce_scatter(tensor):
    rank = dist.get_rank()
    world_size = dist.get_world_size()

    chunks = [chunk.clone() for chunk in tensor.chunk(world_size)]

    send_idx = (rank - 1) % world_size

    for _ in range(world_size - 1):
        recv_idx = (send_idx - 1) % world_size

        send_req = dist.isend(
            chunks[send_idx],
            dst=(rank + 1) % world_size,
        )

        recv = torch.empty_like(chunks[recv_idx])
        dist.recv(
            recv,
            src=(rank - 1) % world_size,
        )

        send_req.wait()

        chunks[recv_idx] += recv
        send_idx = recv_idx

    return chunks[rank]


def ring_all_gather(shard):
    rank = dist.get_rank()
    world_size = dist.get_world_size()

    chunks = [torch.empty_like(shard) for _ in range(world_size)]
    chunks[rank].copy_(shard)

    for step in range(world_size - 1):
        send_idx = (rank - step) % world_size
        recv_idx = (rank - step - 1) % world_size

        send_req = dist.isend(
            chunks[send_idx],
            dst=(rank + 1) % world_size,
        )

        dist.recv(
            chunks[recv_idx],
            src=(rank - 1) % world_size,
        )

        send_req.wait()

    return torch.cat(chunks)

def ring_all_reduce(tensor):
    shard = ring_reduce_scatter(tensor)
    return ring_all_gather(shard)


## Сравнение с PyTorch

Сравним нашу реализацию с готовыми функциями торча `dist.reduce_scatter_tensor`, `dist.all_gather`
и `dist.all_reduce` на нескольких размерах тензоров. 

In [1]:
%%writefile check.py
from datetime import timedelta

import torch
import torch.distributed as dist

from collectives import ring_all_gather, ring_all_reduce, ring_reduce_scatter


torch.set_num_threads(1)
dist.init_process_group("gloo", timeout=timedelta(seconds=30))
try:
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    generator = torch.Generator().manual_seed(42 + rank)

    for chunk_size in (1, 7, 31):
        values = torch.randn(world_size * chunk_size, generator=generator, dtype=torch.float64)
        original = values.clone()
        expected_shard = torch.empty(chunk_size, dtype=values.dtype)
        dist.reduce_scatter_tensor(expected_shard, values.clone(), op=dist.ReduceOp.SUM)
        torch.testing.assert_close(ring_reduce_scatter(values), expected_shard)
        torch.testing.assert_close(values, original, rtol=0, atol=0)

        shard = torch.randn(chunk_size, generator=generator, dtype=torch.float64)
        original_shard = shard.clone()
        expected_parts = [torch.empty_like(shard) for _ in range(world_size)]
        dist.all_gather(expected_parts, shard)
        torch.testing.assert_close(ring_all_gather(shard), torch.cat(expected_parts))
        torch.testing.assert_close(shard, original_shard, rtol=0, atol=0)

        expected_sum = values.clone()
        dist.all_reduce(expected_sum, op=dist.ReduceOp.SUM)
        torch.testing.assert_close(ring_all_reduce(values), expected_sum)
        torch.testing.assert_close(values, original, rtol=0, atol=0)

    if rank == 0:
        print(f"OK: все три операции прошли проверки на {world_size} процессах.")
finally:
    dist.destroy_process_group()

Writing check.py


In [ ]:
import sys

!{sys.executable} -m torch.distributed.run --standalone --nproc-per-node=2 check.py

In [ ]:
!{sys.executable} -m torch.distributed.run --standalone --nproc-per-node=4 check.py